In [ ]:
# import torch; assert torch.cuda.is_available()
# %pip install medicalmultitaskmodeling m3-sdk

In [ ]:
# @title Utilities, Setup, Data Download
import os
import uuid
import wandb
from pathlib import Path
import logfire
import torch
import numpy as np
import math
from PIL import Image
import matplotlib.pyplot as plt
import zipfile
import torchvision.transforms.functional as F

# Disable Logfire integration if no token is set
logfire.configure(send_to_logfire="if-token-present", sampling=logfire.SamplingOptions.level_or_duration())

if "WANDB_API_KEY" not in os.environ:
    print("WANDB_API_KEY not found in environment variables. W&B logging will not be used.")
    os.environ["WANDB_MODE"] = "offline"  # Remove this to use M3's W&B integration
    # # Without W&B the folder for logging predictions can be set manually
    # if "MMM_default_log_folder" not in os.environ:
    #     Path(local_predictions_dir := "./local_predictions").mkdir(parents=True, exist_ok=True)
    #     print(f"Logging predictions to {local_predictions_dir}")
    #     os.environ["MMM_default_log_folder"] = local_predictions_dir

# Download demo data
(DATA_ROOT := Path(os.getenv("ML_DATA_CACHE", default="./data"))).mkdir(parents=True, exist_ok=True)
if not (dataset_path := DATA_ROOT / "bloodmnist_128.npz").exists():
    torch.hub.download_url_to_file(
        "https://cloud.lotz.dev/public.php/dav/files/Zi4mqa7A7Z2Hnsk/?accept=zip", str(dataset_path.absolute())
    )
if not (dataset_path_3d := DATA_ROOT / "organmnist3d_64.npz").exists():
    torch.hub.download_url_to_file(
        "https://cloud.lotz.dev/public.php/dav/files/sQcgeA8cR2XP54w/?accept=zip", str(dataset_path_3d.absolute())
    )

def visualize(cases):
    """
    Visualizes a list of cases by plotting their images and class names in a grid.
    Each case is expected to be a dictionary with an "image" key containing the image data
    and a "meta" key containing a dictionary with a "caption" key for the caption.
    """
    nrows, ncols = math.ceil(math.sqrt(len(cases))), math.ceil(math.sqrt(len(cases)))
    fig, axes = plt.subplots(nrows, ncols)
    images = [case["image"] for case in cases]
    captions = [case["meta"]["caption"] for case in cases]
    overlays = [case.get("label", None) for case in cases]
    for ax, img, cap, overlay in zip(axes.flat, images, captions, overlays):
        ax.imshow(img)
        if overlay is not None:
            ax.imshow(overlay, alpha=0.5)
        ax.set_title(str(cap), fontsize=10)
        ax.axis("off")
    
    plt.show()

In [ ]:
# Preparing the environment.
# %env LOCAL_DEV_ENV=True                   # For Jobs running on a cluster, the variable should be set to False For interactive Notebooks, True.
# %env MMM_LICENSE_ACCEPTED=i accept        # Ofc you accept the MMM license agreement.

# This one-liner imports MMM utilities relevant to interactive programming.
from mmm.interactive import configs as cfs, data, tasks, training, pipes, blocks, api

In [ ]:
# Most experiments need the same configurations with only slight changes.
# This prepares for multi-node multi-gpu training if torchrun is used.
env = cfs.EnvByConvention("finetuning").if_torchrun_prepare()

## Config

In [ ]:
from mmm.api.M3Model import UNICORN_ENCODER, M3_MODELS, DEFAULT_MODEL, WSC_MTL_TINY

# By convention, we collect the parameters that should be configurable without code changes into a HyperParameters object
# All objects in the library have config options implemented with pydantic
# If you want to examine the impact of different learning rates (for example), you can change the config for different jobs.
class HyperParameters(cfs.ExperimentHyperParameters):
    foundation_model: data.DistributedPath | str = M3_MODELS[UNICORN_ENCODER]

    experiment_name: str = "finetuning_demo"                        # So you know what run you are dealing with
    resumable: bool = True                                          # Whether to resume from an existing checkpoint if available

    trainer: training.MTLTrainer.Config = training.MTLTrainer.Config(
        checkpoint_cache_folder=Path("trainer_checkpoints"),        # by default, config->result=true jobs will be resumed
        train_device="cuda",                                        # "cuda" or "cpu"
        mtl_train_loop=cfs.TrainLoopConfig(max_steps=200),          # For experimentation, the steps per loop can be reduced. -1 is a full iteration over all samples.
        mtl_val_loop=cfs.ValLoopConfig(max_steps=100),                    
        max_epochs=3,                                               # Number of epochs to run the training. 3 for Demonstration.
    )

In [ ]:
HyperParameters.update_schema(env)

In [ ]:
# In interactive environments the config is loaded from a file that is always located at ./job_configs/env_name.jsonc
# VSCode provides auto-completion for all configuration options via JSON schema. if does not exist, it will be created.
config = HyperParameters.load_config(env)

## Anatomy of a multi-task model

- Our multi-task models consist of shared blocks (see `blocks.SharedBlock`) and tasks (see `tasks.MTLTask`)
- All blocks and tasks are PyTorch modules
- We start with 2D classification. A classification task requires
  - a `blocks.PyramidEncoder` which transforms an image Tensor[C, H, W] into feature maps list[Tensor[C, H, W]]
  - a `blocks.Squeezer` which transforms the feature maps list[Tensor[C, H, W]] into a latent representation Tensor[C, H, W]
  - a `tasks.ClassificationTask` which takes a latent representation, make predictions, and visualizes results

In [ ]:
# Download the model into the directory specified by env variable ML_DATA_CACHE, otherwise ~/.mmm/
foundation_model = api.M3Model(config.foundation_model, device_identifier=config.trainer.train_device)

# The model can be used as a dictionary of modules.
encoder: blocks.PyramidEncoder = foundation_model["encoder"]
squeezer: blocks.Squeezer = foundation_model["squeezer"]
encoder.freeze_all_parameters()  # freeze the encoder, only other modules (partial fine-tuning)

In [ ]:
# Inputs are batches of images like (batch_size, channels, height, width) which are between 0 and 1.
with torch.no_grad():
    test_input = torch.rand(2, 3, 64, 64).to(encoder.torch_device)
    feature_maps = encoder(test_input)
    latent_feature_map, latent_representation = squeezer(feature_maps)
    print("\n".join([f"{feat_map.shape}" for feat_map in feature_maps]), f"\nLatent: {latent_representation.shape}")

## Logging

For confidential data you should use an internal WANDB instance using `%env WANDB_BASE_URL=http://your-host:PORT/`
For logging to the official servers (including some of your training images by default), create an account at https://wandb.ai/ and be ready to paste your key into here.

The logging is integrated with our config system. All your user-configurable settings should be visible in the overview of the respective experiment:

In [ ]:
# os.environ["WANDB_MODE"] = "offline"  # If you don't want to log to W&B, set the environment variable to "offline"
wandb_run = config.init_experiment(env)

## Preparing data

**Download**

In this guide, we will start a simple classification fine-tuning with the https://medmnist.com/ database.
The `bloodmnist` and `organmnist3D` have been downloaded already in the first cell of this notebook. You can install more available data from here: https://zenodo.org/records/10519652.

**Conventions**

MMM uses [PyTorch dataloading](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) where each sample is a dictionary with fixed keys. The MedMNIST dataset has a fixed length. In consequence, we will use a `torch.utils.data.Dataset` to wrap it.

In [ ]:
class MedMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, class_names):
        self.images, self.labels, self.class_names = images, labels, class_names

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict:
        class_idx = self.labels[index].item()
        # Arbitrary info may only be given under the "meta" key
        return {"image": self.images[index], "class": class_idx, "meta": {"caption": self.class_names[class_idx]}}

class_names = [
    "basophil",
    "eosinophil",
    "erythroblast",
    "immature granulocytes",
    "lymphocyte",
    "monocyte",
    "neutrophil",
    "platelet",
]

# load the downloaded dataset
blooddata = np.load(dataset_path)

# create train and validation datasets
train_dataset = MedMNISTDataset(blooddata["train_images"], blooddata["train_labels"], class_names=class_names)
val_dataset = MedMNISTDataset(blooddata["val_images"], blooddata["val_labels"], class_names=class_names)

# visualize some samples to have a look at the data
visualize([train_dataset[i] for i in range(16)])

Using a `TrainValCohort` you tell MMM which data to use for training and which for validation. It is also the place to define data augmentation.

In [ ]:
def transform_pil_to_mmm(case: dict) -> dict:
    """
    MMM datasets are wrappers around PyTorch datasets that require a very specific format for each type of label.
    For classification, a dictionary with "image" and "class" keys is expected.
    """
    return {"image": F.to_tensor(Image.fromarray(case["image"]).convert("RGB")), "class": case["class"]}


def mmm_cohort(train_dataset, val_dataset) -> data.TrainValCohort:
    """
    The data is encapsulated in an object that holds a training and a validation set.
    This Object is Called a TrainValCohort. It is initialized with MMM Classification datasets,
    which are compatible with a MMM classification task.

    More info such as class names was taken from https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py
    """
    # This Object also has a config to set hyperparameters, such as batch size for train and validation set 
    train_val_config = data.TrainValCohort.Config(batch_size=(8, 8), num_workers=2)

    # the torch.utils.Dataset is wrapped in a MMM ClassificationDataset.
    train_ds = data.ClassificationDataset(
            train_dataset,                                              # The torch.utils.Dataset is wrapped in a MMM ClassificationDataset.
            src_transform=transform_pil_to_mmm,                         # function that should be applied to every case not matter what
            batch_transform=pipes.Alb(pipes.get_weak_default_augs()),   # Augmentations are applied only to training.
            class_names=class_names,                                    # class names for logging 
        )
    
    # Do the same for the validation set, but without augmentations
    val_ds = data.ClassificationDataset(val_dataset, src_transform=transform_pil_to_mmm, class_names=class_names)

    # return the TrainValCohort with the datasets and the config
    return data.TrainValCohort(
        args=train_val_config,
        train_ds=train_ds,
        val_ds=val_ds,
    )

# Pass the torch.utils.Dataset to the TrainValCohort to make them mmm compatible.
mmm_train_dataset = (my_cohort := mmm_cohort(train_dataset, val_dataset)).datasets[0]
mmm_training_case = mmm_train_dataset[0]
mmm_training_case["image"].shape, mmm_training_case["class"]

## Training your model

`training.MTLTrainer` is responsible for running the (multi-task) training loop. By default, it uses gradient accumulation to perform update steps consisting of all tasks that were added using `trainer.add_mtl_task(...)`. By default, it starts with a validation loop and runs each loop until exhaustion. The last step of a task might consist of a batch with smaller batchsize than the other steps.

In [ ]:
trainer: training.MTLTrainer = training.MTLTrainer(
    config.trainer,
    experiment_name=cfs.remove_wandb_special_chars(config.experiment_name),
    clear_checkpoints=not config.resumable,
)
# The trainer needs to know about the encoder, squeezer, decoder, and Grouper id they ought to be used.
trainer.add_shared_blocks([foundation_model[k] for k in foundation_model.get_sharedblock_keys()])

Each `tasks.MTLTask` assembles its own architecture consisting of its own modules and the shared blocks. In the case of the `tasks.ClassificationTask`, these shared blocks are the shared encoder and the shared squeezer. Each `tasks.MTLTask` needs a unique name.

In [ ]:
mmm_task = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    args=tasks.ClassificationTask.Config(module_name="Example_MNIST_blood_classification"),
    cohort=my_cohort,
)

trainer.add_mtl_task(mmm_task)
print("added task")

In [ ]:
# After adding all tasks, the trainer is ready for training.
trainer.fit()

# Try it with your own Data

This is the point where you could run the pipeline with your own data!

Or continue to see how to deal with 3D data and an image encoder that is trained on 2D images.

## 3D data Fine-tuning

In [ ]:
from mmm.volume3d import Tomo3DProcessor

classes_3d = [
    "liver",
    "kidney-right",
    "kidney-left",
    "femur-right",
    "femur-left",
    "bladder",
    "heart",
    "lung-right",
    "lung-left",
    "spleen",
    "pancreas",
]
data3d = np.load(dataset_path_3d)
train_3d = MedMNISTDataset(data3d["train_images"], data3d["train_labels"], class_names=classes_3d)
val_3d = MedMNISTDataset(data3d["val_images"], data3d["val_labels"], class_names=classes_3d)

def transform_volume_to_slices(case: dict):
    npy_volume, label = case["image"], case["class"]
    npy_volume = npy_volume.astype(np.float32) / 255.0  # Normalize to [0, 1]
    patient_id = uuid.uuid4().hex[:8]  # Generate a random patient ID for grouping slices
    return [
        {
            # All image inputs are expected to be 3-channel
            "image": Tomo3DProcessor.repeat_channels(torch.from_numpy(npy_volume[..., i]).unsqueeze(0)).float(),
            "class": label,
            # In order to know which slices belongs to which ID, 
            # the "meta" key should contain a "group_id" that is the same for all slices of one patient 
            # and different for different patients in the batch.
            "meta": {"group_id": patient_id},
        }
        for i in range(npy_volume.shape[-1])
    ]

In [ ]:

def mmm_cohort_3d(train_dataset, val_dataset) -> data.TrainValCohort:
    return data.TrainValCohort(
        data.TrainValCohort.Config(batch_size=(2, 2), num_workers=2),  # 2 train, 2 val
        train_ds=data.ClassificationDataset(
            train_dataset,
            src_transform=transform_volume_to_slices,
            batch_transform=pipes.ApplyToList(pipes.Alb(pipes.get_weak_default_augs(), replay_for_groups=True)),
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate,
        ),
        val_ds=data.ClassificationDataset(
            val_dataset,
            src_transform=transform_volume_to_slices,
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate,
        ),
    )


mmm_train_dataset_3d = (my_cohort := mmm_cohort_3d(train_3d, val_3d)).datasets[0]

mmm_task_3d = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    # Instruct the task to use the "grouper" transformer module from the foundation model to enable 3D context
    args=tasks.ClassificationTask.Config(
        module_name="3d_organ_classification",
        # This tells the Task to use the grouper module in order to aggregate the individual 2D slices.
        grouper_key=cfs.GroupUsage(grouper_key="grouper")
    ),
    cohort=mmm_cohort_3d(train_3d, val_3d),
)

# To not automatically resume the previous training, we change the experiment name.
# Otherwise, the trainer automatically check if the experiment Name is present and load the checkpoint including tasks.
# This is helpful if you are working on a cluster with multiple people and your training job got killed.
config.experiment_name = "finetuning_demo_3d"
trainer: training.MTLTrainer = training.MTLTrainer(
    config.trainer,
    experiment_name=cfs.remove_wandb_special_chars(config.experiment_name),
    clear_checkpoints=not config.resumable,
)
# The trainer needs to know about the encoder, squeezer, decoder, and Grouper id they ought to be used.
trainer.add_shared_blocks([foundation_model[k] for k in foundation_model.get_sharedblock_keys()])

trainer.add_mtl_task(mmm_task_3d)
print(f"added task {mmm_task_3d.get_name()}")

In [ ]:
trainer.fit()

## Exporting the model

For different use cases we recommend different export methods:

- Native PyTorch export via `MTLTrainer.save_blocks_native`. This exports an `nn.ModuleDict` object which is expected by our inference utilities. This has the disadvantage that all dependencies have to be installed exactly as they were during the export because this uses `pickle` internally. This method is recommended when you have control over the inference environment (e.g. by using the same container as during training).
- ONNX export via `SharedBlock.export_to_onnx`. This exports a single shared block such as the encoder using the established ONNX standard. This is good for sharing with external users.

In [ ]:
module_dict = trainer.save_blocks_native(
    export_modules_path := data.DistributedPath.from_string("./all_blocks.pt.zip"),
    only_inference=True,  # Cohorts often should not be exported, `only_inference` strips those.
)
module_dict.keys()

In [ ]:
# Loading requires only one line of torch and is not specific to MMM:
exported_dict = api.M3Model(export_modules_path, "cuda:0")
with torch.inference_mode():
    some_feature_maps = exported_dict["encoder"](test_input)
    print(some_feature_maps[-1].shape)

The native PyTorch export requires the user to know how to assemble the blocks together. Alternatively, native export of individual tasks can be used to export a whole task's pipeline including the shared blocks. For this, the `save_task_native` of `MTLTrainer` can be used.

In [ ]:
trainer.save_task_native("3d_organ_classification", Path("./task.pt"), only_inference=True)

In [ ]:
val_3d_dataset = mmm_cohort_3d(train_3d, val_3d).datasets[1]
test_case = val_3d_dataset[0]

fig, axs = plt.subplots(1,5, figsize=(15,5))
for i in range(5):
    axs[i].imshow(test_case[i]["image"].permute(1,2,0).squeeze().numpy(), cmap="gray")
    axs[i].set_title(f"Class: {classes_3d[test_case[i*2]['class']]}")
    axs[i].axis("off")

In [ ]:
# Loading again only requires one line and is not specific to MMM:
exported_task = torch.load("./task.pt", weights_only=False)

# prepare the test_case to be one tensor of shape (N, channels, height, width) on the same device as the task
test_input = torch.stack([t["image"] for t in test_case]).to(exported_task.task.torch_device)

with torch.inference_mode():
    # The task module returns the logits, torch.argmax is used to get the class index for classification.
    task_output = exported_task.forward((test_input, supercase_indexes := torch.zeros(len(test_input)).long().to(exported_task.task.torch_device)))

# We obtained a class prediction for every slice in the test volume.
# to aggregate we use a majority voting of the output.
predicted_class_index = torch.argmax(torch.argmax(task_output, dim=1).cpu().bincount())
mmm_task_3d.class_names[predicted_class_index], predicted_class_index.item()

# Exercise
You have reached the end of the guided tour. If you want, you can think of a way to fine-tune the FM with multiple different tasks. At last, this is a multi-task learning pipline. How would a Segmentation task fit in? Can you train a model with 1 classification, 1 3D classification, and 1 segmentation task?

In [ ]:
segmentation_data_folder = Path("segmentationdata")
torch.hub.download_url_to_file("https://owncloud.fraunhofer.de/index.php/s/UvxtT0B2Q9CsicY/download", str(segmentation_data_folder.with_suffix(".zip")))
# unzip the downloaded file
with zipfile.ZipFile(segmentation_data_folder.with_suffix(".zip"), "r") as zip_ref:
    zip_ref.extractall(segmentation_data_folder)

def find_image_for_mask(mask_path: Path) -> Path:
    return [p for p in [mask_path.with_suffix(".png"), mask_path.with_suffix(".jpg")] if p.exists()][0]

# mask: -1 is unlabeled, 0 is background, 1 is foreground
segmentation_cases = [
    {
        "image": Image.open(find_image_for_mask(mask_path)).convert("RGB"),
        "label": np.load(mask_path),
        "meta": {"mask_path": str(mask_path), "caption": f"{mask_path.stem}"}
    }
    for mask_path in (segmentation_data_folder / "masks").glob("*.npy")
]
visualize(segmentation_cases)

In [ ]:
# from torchvision.transforms import Resize
# from torch.utils.data import Dataset
# from copy import deepcopy
# from cv2 import resize, INTER_NEAREST

# def to_mmm_convention(d: dict):
#     resizer = Resize(size=(224, 224))
#     # ...
#     return d

# seg_task = tasks.SemSegTask(
#     class_names=["background", "foreground"],
#     for_decoder=foundation_model["decoder"],
#     for_squeezer=foundation_model["squeezer"],
#     args=tasks.SemSegTask.Config(module_name="lung_segmentation"),
#     cohort=seg_cohort
# )
